# Build the total DL1 datacheck table

Builds one subrun-wise table (`df_flat`) from all DL1 datacheck HDF5 files, adding:
* DL1/DL2/DL3 paths
* Weather-station (WS)
* Intensity cuts
* And the nearest AMC PSF measurement.
Finishes by aggregating everything to a run-wise table (`df_flat_runwise`).

Long-running steps (WS-run matching, intensity cuts, PSF queries) are submitted as
SLURM jobs and backups are stored.

In [ ]:
import re
import warnings
from pathlib import Path
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from datetime import datetime
import numpy as np
import pandas as pd
from scipy.optimize import OptimizeWarning
pd.set_option("display.max_columns", None)

import utils

warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=OptimizeWarning)

## Configuration

Every tunable parameter, flag, and path lives here. Nothing below this cell
should hardcode a path or a magic number.

In [ ]:
# ── Toggle which stages to (re-)compute ─────────────────────────────────────
COMPUTE_DCHECK   = True    # re-read all HDF5 files and rebuild the flat table
COMPUTE_WS       = True    # re-load WS day files from disk
ADD_WS_DATA      = True    # merge WS into df_flat after the SLURM matching step
PROCESS_INLINE   = False   # False -> submit to SLURM; True -> run in-process (debug)
OVERWRITE        = False    # ignore all on-disk / cache-file shortcuts

# ── SLURM batch sizes ────────────────────────────────────────────────────────
N_ROWS_PER_JOB      = 6_000   # subruns per WS-matching SLURM job
RUNS_PER_JOB        = 20      # runs per intensity-cut SLURM job
QUERY_BATCH_MONTHS  = 3       # months per PSF-query SLURM job
SLURM_RESUME_FROM   = 0       # set > 0 to resume after a failed run; 0 = start fresh

# ── Directory layout ────────────────────────────────────────────────────────
ROOT       = Path.cwd()
ROOT_DATA  = ROOT / "data"
ROOT_TMP   = ROOT_DATA / "tmp"
PLOT_DIR   = ROOT / "plots"

DCHECK_DIR    = ROOT_DATA / "datachecks"
WS_DIR        = ROOT_DATA / "ws"
PSF_DIR       = ROOT_DATA / "psf"
PSF_BATCH_DIR = PSF_DIR / "batches"
CUTS_DIR      = ROOT_TMP / "intensity_cuts"
SLURM_OUT_DIR = ROOT_DATA / "slurm_output"
SLURM_B3      = ROOT_DATA / "brightest3"
SCRIPT_PATH   = ROOT / "script_datachecks.py"

for d in [DCHECK_DIR, WS_DIR, PSF_BATCH_DIR, CUTS_DIR, SLURM_OUT_DIR, PLOT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

# ── Output / cache files ──────────────────────────────────────────────────────
FNAME_DCHECK_FLAT_SRUNWISE = DCHECK_DIR / "datachecks_flat.parquet"
FNAME_DCHECK_FLAT_RUNWISE = DCHECK_DIR / "datachecks_flat_runwise.parquet"
FNAME_DCHECK_PROCESSED = DCHECK_DIR / "processed_files.txt"

FNAME_B3_DATA = SLURM_B3 / "srunwise_ly_b3_crab_correction.ecsv"

FNAME_WS_REDUCED = WS_DIR / "reduced_ws_plus_dates.pkl"
FNAME_WS_RUN_REL = WS_DIR / "ws_run_relation.txt"

FNAME_PSF_RAW   = PSF_DIR / "psf_amc_spotsize.txt"    # final, deduped cache
FNAME_PSF_STATE = PSF_DIR / "psf_query_state.txt"     # last tstop fully committed to cache
VAR_TCU         = "AMC_PSF_Spotsize"

# ── Source directories / glob patterns ────────────────────────────────────────
ROOT_DL1_D_OLD = Path("/fefs/aswg/data/real/DL1/datacheck_files/night_wise")
ROOT_DL1_D_NEW = Path("/fefs/onsite/data/lst-pipe/LSTN-01/DL1/datacheck_files/night_wise")

ROOT_DL1_OLD = "/fefs/aswg/data/real/DL1/????????/v*/tailcut*/dl1_LST-1.Run?????.????.h5"
ROOT_DL1_NEW = "/fefs/onsite/data/lst-pipe/LSTN-01/DL1/????????/v*/tailcut*/dl1_LST-1.Run?????.????.h5"
ROOT_DL2_OLD = "/fefs/aswg/data/real/DL2/????????/v*/tailcut*/nsb_tuning_*/dl2_LST-1.Run?????.h5"
ROOT_DL2_NEW = "/fefs/onsite/data/lst-pipe/LSTN-01/DL2/????????/v*/tailcut*/nsb_tuning_*/dl2_LST-1.Run?????.h5"
ROOT_DL3_NEW = (
    "/fefs/onsite/data/lst-pipe/LSTN-01/DL3/????????/v*/tailcut*/nsb_tuning_*/*/std/"
    "*allsky*/src_indep/point/ring-wobble/gheff*_thetacont*/irf_interp/dl3_LST-1.Run?????.fits"
)

PATH_WS = Path("/fefs/aswg/workspace/juan.jimenez/data/ws_magic/uncompressed/")

# ── Reference power-law parameters (ZD-corrected cosmics rate fit) ────────────
REF_P0, REF_P1 = 1.74, -2.23

## Step 1 - Build the subrun-wise datacheck table

Reads every DL1 datacheck HDF5 file once (list stored in `processed_files.txt` so only reads back new files), merges its cosmics / flatfield / pedestal tables, and stores the result as a single flat table in parquet format.

Here the colums should be:
* Run basic information, numbers, times, pointings (ra, dec) and number of events
* Tailcut (pict/bound) and NSB
* FF events, number, time and charge info
* Pedestal events, number, charge, fractions
* Cosmics rates
* Intensity spectrum parameterization
* Intensity spectrum parameterization ZD corrected + Light Yield (intensity)
* Muon rings, number, efficiency, width, radius, intensity, hg_peak
* Datacheck filename

In [ ]:
OVERWRITE = True

In [ ]:
%%time
if "df_flat" in globals() and not OVERWRITE:
    print(f"--> [MEMORY] df_flat already loaded ({len(df_flat):,} rows). Skipping.")

elif COMPUTE_DCHECK:
    all_files = sorted(ROOT_DL1_D_OLD.glob("*.h5")) + sorted(ROOT_DL1_D_NEW.glob("*.h5"))
    print(f"Found {len(all_files)} total datacheck files on disk.")

    already_done = (
        set(FNAME_DCHECK_PROCESSED.read_text().splitlines())
        if FNAME_DCHECK_PROCESSED.exists() and FNAME_DCHECK_FLAT_SRUNWISE.exists() and not OVERWRITE
        else set()
    )
    new_files = [p for p in all_files if str(p.resolve()).strip() not in already_done]
    print(f"Already cached : {len(already_done)} file(s)")
    print(f"New to read    : {len(new_files)} file(s)")

    if new_files:
        new_frames, processed_now = [], []
        for i, path in enumerate(new_files):
            print(f"  Reading {i + 1}/{len(new_files)}: {path.name}", end="\r")
            df_one = utils.read_dcheck_file(path)
            processed_now.append(str(path.resolve()).strip())
            if df_one is not None:
                new_frames.append(df_one)

        df_existing = (
            pd.read_parquet(FNAME_DCHECK_FLAT_SRUNWISE)
            if FNAME_DCHECK_FLAT_SRUNWISE.exists() and not OVERWRITE
            else pd.DataFrame()
        )

        if new_frames:
            df_new = utils.fix_dcheck_columns(pd.concat(new_frames, ignore_index=True))
            # Rename BEFORE concatenating with the cached (already-renamed) df_existing,
            # otherwise concat keeps both 'runnumber' and 'obs_id' as separate columns
            # and the rename below then creates a duplicate 'obs_id' column.
            df_new = df_new.rename(columns={"runnumber": "obs_id"})
            df_flat = pd.concat([df_existing, df_new], ignore_index=True) if not df_existing.empty else df_new
            print(f"\n\nAdded {len(df_new):,} new subrun rows.")
        else:
            df_flat = df_existing
            print("\n[WARN] No new data frames could be extracted from these files.")

        if not df_flat.empty:
            df_flat["dcheck_fname"] = df_flat["dcheck_fname"].astype("category")

            df_flat = utils.order_dcheck_columns(df_flat)
            df_flat = df_flat.sort_values(["obs_id", "subrun"], ignore_index=True)
            df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)

        FNAME_DCHECK_PROCESSED.write_text("\n".join(sorted(already_done | set(processed_now))) + "\n")
    else:
        print("--> [DISK] Nothing new. Loading full dataset from Parquet cache...")
        df_flat = pd.read_parquet(FNAME_DCHECK_FLAT_SRUNWISE)

    print(f"Total: {len(df_flat):,} subrun rows.")

else:
    print("--> [DISK] COMPUTE_DCHECK is False. Loading from Parquet...")
    df_flat = pd.read_parquet(FNAME_DCHECK_FLAT_SRUNWISE)

df_flat.head(3)

In [ ]:
OVERWRITE = False

### Step 1b - Adding the light yield but computed locally + uncertainty
Adding 2 new columns:
* ly_i
* ly_i_err

In [ ]:
ly_i, ly_i_err = utils.calc_light_yield(
    df_flat["ZD_corrected_cosmics_rate_at_422_pe"], 
    df_flat["ZD_corrected_cosmics_spectral_index"], 
    df_flat["ZD_corrected_delta_cosmics_rate_at_422_pe"], 
    df_flat["delta_cosmics_spectral_index"], 
    REF_P0
)

df_flat["ly_i"] = ly_i
df_flat["ly_i_err"] = ly_i_err

### Step 1c - Resolve DL1/DL2/DL3 file paths per run

Searching in the "new" and "old" paths, but proprizing the new one. The subrun-wise DL1 path is derived from the run-wise one by swapping in the run/subrun string. Here we add the following columns ofr the paths:
* dl1_subrun, dl1_run
* dl2_run
* dl3_run

In [ ]:
%%time
print(f"Resolving file paths for {len(df_flat):,} rows...")
unique_runs = df_flat["obs_id"].unique()

dl1_map = utils.resolve_paths_by_run(unique_runs, ROOT_DL1_NEW, ROOT_DL1_OLD, subrun_glob="Run?????.????.h5")
dl2_map = utils.resolve_paths_by_run(unique_runs, ROOT_DL2_NEW, ROOT_DL2_OLD, subrun_glob="Run?????.h5")
dl3_map = utils.resolve_paths_by_run(unique_runs, ROOT_DL3_NEW, None,         subrun_glob="Run?????.fits")

df_flat["dl1_runwise_fname"] = df_flat["obs_id"].map(dl1_map)
df_flat["dl2_fname"] = df_flat["obs_id"].map(dl2_map)
df_flat["dl3_fname"] = df_flat["obs_id"].map(dl3_map)

# Subrun-wise DL1 path: swap the run/subrun string into whichever DL1 path was resolved above
df_flat["dl1_srunwise_fname"] = [
    re.sub(r"Run\d+\.\d+\.h5", f"Run{int(r):05d}.{int(s):04d}.h5", p) if pd.notna(p) else None
    for r, s, p in zip(df_flat["obs_id"], df_flat["subrun"], df_flat["dl1_runwise_fname"])
]

for col in ["dl1_runwise_fname", "dl1_srunwise_fname", "dl2_fname", "dl3_fname"]:
    df_flat[col] = df_flat[col].astype("category")

df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)
print("Paths resolved and cached.")

## Step 2 - Load WS data

Each day lives in `PATH_WS/<YYYY_MM>/WSalldata_<YYMMDD>.txt`. We load only the
days overlapping the datacheck date range (see `utils.load_ws_day`).

In [ ]:
%%time
if COMPUTE_WS:
    t_min, t_max = pd.Timestamp(df_flat["time"].min()), pd.Timestamp(df_flat["time"].max())
    days = pd.date_range(start=t_min.floor("D"), end=t_max.ceil("D"), freq="D")
    print(f"[WS] Date range: {t_min.date()} -> {t_max.date()}  ({len(days)} day file(s))")

    frames, n_missing, bad_days = [], 0, []
    for i, day in enumerate(days, 1):
        try:
            result = utils.load_ws_day(day, PATH_WS)
        except Exception as e:
            print(f"\n[WS] FAILED on {day.date()}: {type(e).__name__}: {e}")
            bad_days.append((day, e))
            continue  # skip this file and keep going

        if result is not None:
            frames.append(result)
        else:
            n_missing += 1
        print(f"  [{i}/{len(days)}] {'*' if result is not None else '-'} {day.date()}", end="\r")

    print(f"\n[WS] Loaded {len(frames)} file(s), {n_missing} missing, {len(bad_days)} failed.")
    if bad_days:
        print("[WS] Days that failed to parse:")
        for day, e in bad_days:
            print(f"   - {day.date()}: {e}")

    if not frames:
        raise FileNotFoundError("No WS files found in the datacheck date range.")

    df_ws = pd.concat(frames).sort_index()
    df_ws = df_ws[(df_ws.index >= t_min) & (df_ws.index <= t_max)]
    print(f"[WS] {len(df_ws):,} rows retained after masking.")

    pd.to_pickle([df_ws, df_ws.index.to_numpy()], FNAME_WS_REDUCED)
    print(f"[WS] Saved -> {FNAME_WS_REDUCED}")
else:
    df_ws, _dates_ws = pd.read_pickle(FNAME_WS_REDUCED)
    print(f"[WS] {len(df_ws):,} rows loaded from cache.")

## Step 3 - Match subruns to the nearest WS timestamp (SLURM)

Delegates the actual nearest-timestamp lookup to `script_datachecks.py
write_ws_run_relation`, split into row-index batches so it can run in
parallel on SLURM (or inline for debugging).

In [ ]:
%%time
batches = utils.chunk_row_batches(len(df_flat), N_ROWS_PER_JOB)
print(f"Total subruns : {len(df_flat):,}  |  Subruns/job : {N_ROWS_PER_JOB:,}  |  Jobs prepared : {len(batches)}")

def _ws_job_name(idx, batch):
    i1, i2 = batch
    return f"ws_run_rel_{i1}_{i2}"

def _ws_cmd(batch):
    i1, i2 = batch
    args = [str(i1), str(i2), str(FNAME_DCHECK_FLAT_SRUNWISE), str(FNAME_WS_REDUCED), str(FNAME_WS_RUN_REL)]
    return f"python {SCRIPT_PATH} write_ws_run_relation {' '.join(args)}"

processed, skipped = utils.run_batches(
    batches, _ws_job_name, _ws_cmd, SLURM_OUT_DIR,
    process_inline=PROCESS_INLINE, resume_from=SLURM_RESUME_FROM, overwrite=OVERWRITE,
)
print(f"\n--- RUN SUMMARY --- processed: {processed} | skipped: {skipped}")

---
> **Wait here until all SLURM jobs complete before continuing.**
>
> Monitor with `!squeue -u $USER`, cancel with `!scancel -u $USER`.

In [ ]:
!squeue -u $USER
# !scancel -u juan.jimenez

### Parse and merge WS data into `df_flat`

Dedupes the WS-run relation file (jobs may overlap on resubmission), parses it into a `run -> subrun -> WS timestamp` lookup, then left-merges the matching weather variables into the flat table. The columns that are added in the subrun datacheck:
* Temperature, pressure, humidity
* Wind: speed, gust, avg
* TNG dust and seeing
* Rain

In [ ]:
%%time
n_total, n_unique = utils.dedupe_text_file(FNAME_WS_RUN_REL)
print(f"WS-run relation: {n_total:,} lines -> {n_unique:,} unique (removed {n_total - n_unique:,} duplicates)")

# Parse "run-subrun,date_str" lines into a run -> subrun -> WS-timestamp dict
file_results_lines = np.loadtxt(FNAME_WS_RUN_REL, dtype=str, delimiter=",")
dict_ws_run_rel: dict[int, dict[int, str | None]] = {}
for runsubrun, date_str in file_results_lines:
    run, srun = map(int, runsubrun.split("-"))
    dict_ws_run_rel.setdefault(run, {})[srun] = date_str if date_str != "None" else None
print(f"Parsed {sum(len(v) for v in dict_ws_run_rel.values()):,} subrun entries "
      f"across {len(dict_ws_run_rel):,} runs.")

if ADD_WS_DATA:
    df_dates = pd.DataFrame([
        {"obs_id": run, "subrun": srun, "ws_date_str": str(date_str)}
        for run, sruns in dict_ws_run_rel.items()
        for srun, date_str in sruns.items() if pd.notna(date_str)
    ])
    if "ws_date_str" not in df_flat.columns:
        df_flat = df_flat.merge(df_dates, on=["obs_id", "subrun"], how="left")

    df_ws_clean = df_ws[list(utils.WS_COL_MAP.keys())].rename(columns=utils.WS_COL_MAP)
    df_ws_clean.index = df_ws_clean.index.astype(str)
    df_ws_clean = df_ws_clean[~df_ws_clean.index.duplicated(keep="first")]

    df_flat = df_flat.drop(columns=[c for c in utils.WS_COL_MAP.values() if c in df_flat.columns])
    df_flat = df_flat.merge(df_ws_clean, left_on="ws_date_str", right_index=True, how="left")

    df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)
    print(f"Saved enriched flat table -> {FNAME_DCHECK_FLAT_SRUNWISE}")

## Step 4 - Per-run intensity cut (sending jobs)

Computes one intensity cut per run from its DL2 file (`script_datachecks.py get_intensity_cut`), batching several runs into a single SLURM job to keep the queue short, then broadcasts each run's value to all of its subruns. Then the following column is added:
* intensity_cut

**NOTE: If some runs constantly fail, increase the jobs memory from 10 000 to e.g. 20 000 or 30 000 (there are some long runs...)**

In [ ]:
DONE_DIR = CUTS_DIR / ".done"
DONE_DIR.mkdir(parents=True, exist_ok=True)

def run_marker(run_id) -> Path:
    return DONE_DIR / f"{run_id}.done"

def is_run_done(run_id) -> bool:
    return run_marker(run_id).exists()

valid_dl2_runs = df_flat.dropna(subset=["dl2_fname"]).drop_duplicates("obs_id")
runs_items = list(zip(valid_dl2_runs["obs_id"], valid_dl2_runs["dl2_fname"]))

# filter out already-completed runs BEFORE batching
pending_runs = [(run, fname) for run, fname in runs_items if not is_run_done(run)]
n_done = len(runs_items) - len(pending_runs)
print(f"Total unique runs: {len(runs_items):,} | already done: {n_done:,} | pending: {len(pending_runs):,}")

batches = utils.chunk_list(pending_runs, RUNS_PER_JOB)
print(f"Jobs prepared : {len(batches)}")

def _cut_cmd(batch):
    cmds = [
        f"python {SCRIPT_PATH} get_intensity_cut '{fname}' '{run}' '{CUTS_DIR}' "
        f"&& touch '{run_marker(run)}'"
        for run, fname in batch
    ]
    # ';' not '&&' between runs: one failing run must NOT block the rest of the batch
    return " ; ".join(cmds) + " ; echo 'Finished'"

processed, skipped = utils.run_batches(
    batches, lambda idx, b: f"int_cuts_{idx}", _cut_cmd, SLURM_OUT_DIR,
    process_inline=PROCESS_INLINE, resume_from=SLURM_RESUME_FROM,
    overwrite=True,   # the old .out/.done batch cache is no longer the source of truth
    slurm_opts="--mem=10000 -p short",
)
print(f"\nProcessed: {processed} | Skipped: {skipped}")

In [ ]:
!squeue -u $USER
# !scancel -u juan.jimenez

Adding the data to the datacheck

In [ ]:
%%time
print("Gathering Intensity cuts results from txt files...")
intensity_cut_map = {
    int(f.stem): float(f.read_text().strip()) for f in CUTS_DIR.glob("*.txt")
}
df_flat["intensity_cut"] = df_flat["obs_id"].map(intensity_cut_map)
df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)
print(f"Done! Intensity cuts mapped for {len(intensity_cut_map):,} runs.")

## Step 5 - Query the AMC PSF spot-size

Queries the `AMC_PSF_Spotsize` TCU variable in quarterly batches (state persisted in `psf_query_state.txt` so reruns only fetch new time ranges), dedupes flatline repeats, then matches each run to its nearest PSF reading. The columns added in the subrun wise datacheck:
* PSF size and time where was taken (and time difference in h)

In [ ]:
%%time
psf_start_default = pd.Timestamp(df_flat["time"].min()) - pd.Timedelta(days=7)
psf_stop_target    = min(pd.Timestamp(df_flat["time"].max()) + pd.Timedelta(days=7), pd.Timestamp.now())

if FNAME_PSF_STATE.exists() and not OVERWRITE:
    tstart0 = pd.Timestamp(FNAME_PSF_STATE.read_text().strip())
else:
    tstart0 = psf_start_default
    if OVERWRITE:
        FNAME_PSF_RAW.write_text("")

psf_batches, t = [], tstart0
while t < psf_stop_target:
    t2 = min(t + pd.DateOffset(months=QUERY_BATCH_MONTHS), psf_stop_target)
    psf_batches.append((t, t2))
    t = t2
print(f"[PSF] Query range: {tstart0} -> {psf_stop_target}  |  Batches needed: {len(psf_batches)}")

def _psf_job_name(idx, batch):
    t1, t2 = batch
    return f"psf_{t1.strftime('%Y%m%d')}_{t2.strftime('%Y%m%d')}"
    
def _psf_part_file(idx, batch):
    return PSF_BATCH_DIR / f"{_psf_job_name(idx, batch)}.txt"
    
def _psf_cmd(batch):
    t1, t2 = batch
    part_file = _psf_part_file(0, batch)  # keep consistent with job name
    args = [t1.isoformat(), t2.isoformat(), VAR_TCU, str(part_file)]
    return f"python {SCRIPT_PATH} query_psf_batch {' '.join(args)}"

processed, skipped = utils.run_batches(
    psf_batches, _psf_job_name, _psf_cmd, SLURM_OUT_DIR,
    process_inline=PROCESS_INLINE, resume_from=0, overwrite=OVERWRITE,
    part_file_fn=_psf_part_file,
)
print(f"\n--- RUN SUMMARY --- processed: {processed} | skipped: {skipped}")

In [ ]:
!squeue -u $USER
# !scancel -u juan.jimenez

In [ ]:
from importlib import reload
reload(utils)

Adding those data to the query-state.

In [ ]:
# Concatenate finished batch parts into the raw PSF cache, then dedupe:
part_files = sorted(PSF_BATCH_DIR.glob("psf_*.txt"))
with open(FNAME_PSF_RAW, "a") as f_out:
    for pf in part_files:
        out_path = SLURM_OUT_DIR / f"{pf.stem}.out"
        if utils.is_job_complete(out_path):
            f_out.write(pf.read_text())

# Load and clean
df_psf_raw = pd.read_csv(FNAME_PSF_RAW, header=None, names=["time", "value"])
n_before = len(df_psf_raw)

# --- NEW: Filter out-of-range values ---
df_psf_raw = df_psf_raw[(df_psf_raw["value"] >= 5) & (df_psf_raw["value"] <= 100)]
n_after_filter = len(df_psf_raw)
# --------------------------------------

df_psf_raw["time"] = pd.to_datetime(df_psf_raw["time"], format="mixed")
df_psf_raw = df_psf_raw.drop_duplicates(subset="time").sort_values("time")
n_unique_time = len(df_psf_raw)

# Remove consecutive flatline repeats
df_psf_raw = df_psf_raw[df_psf_raw["value"] != df_psf_raw["value"].shift()]
n_after = len(df_psf_raw)

df_psf_raw.to_csv(FNAME_PSF_RAW, header=False, index=False)

print(f"Total lines parsed         : {n_before:,}")
print(f"Out-of-range removed       : {n_before - n_after_filter:,}")
print(f"Overlapping dups removed   : {n_after_filter - n_unique_time:,}")
print(f"Stagnant flatlines removed : {n_unique_time - n_after:,}")

# Advance the persisted query-state pointer up to the latest *contiguous* completed batch,
# so a partially-finished submission never gets marked done.
committed_tstop = tstart0
for t1, t2 in psf_batches:
    out_path = SLURM_OUT_DIR / f"{_psf_job_name(0, (t1, t2))}.out"
    if utils.is_job_complete(out_path):
        committed_tstop = t2
    else:
        break
FNAME_PSF_STATE.write_text(committed_tstop.isoformat())
print(f"[PSF] Query state advanced to: {committed_tstop}")

Adding the data to the datacheck subrunwise.

In [ ]:
# Run-wise closest-PSF match, broadcast to all subruns of the run
df_psf = pd.read_csv(FNAME_PSF_RAW, header=None, names=["time", "psf_spotsize"])
df_psf["time"] = pd.to_datetime(df_psf["time"])
df_psf = df_psf.drop_duplicates(subset="time").sort_values("time").reset_index(drop=True)

df_run_times = (
    df_flat.groupby("obs_id", as_index=False)["time"].min()
    .assign(time=lambda d: pd.to_datetime(d["time"]))
    .sort_values("time")
)
df_run_psf = pd.merge_asof(
    df_run_times, df_psf.rename(columns={"time": "psf_time"}),
    left_on="time", right_on="psf_time", direction="nearest",
)

df_flat["psf_spotsize"] = df_flat["obs_id"].map(dict(zip(df_run_psf["obs_id"], df_run_psf["psf_spotsize"])))
df_flat["psf_time"] = df_flat["obs_id"].map(dict(zip(df_run_psf["obs_id"], df_run_psf["psf_time"])))
df_flat["psf_time_since_h"] = (pd.to_datetime(df_flat["time"]) - df_flat["psf_time"]).dt.total_seconds() / 3600.0

df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)
print("[PSF] Done! psf_spotsize / psf_time / time_since_psf_hours added to df_flat.")

## Adding the available brightest triplet information
Adding the brightest triplet just to the runs to which it was calculated. Adding the columns:
* LY (from brightest triplet), and error (Forn now 10% error, waiting for real error), and p-value

In [ ]:
%%time
print(f"Loading brightest-triplet (B3) light yield -> {FNAME_B3_DATA}")
tab_b3 = pd.read_csv(FNAME_B3_DATA, sep=",", index_col=0)

# in the future we will have err_ly_b3, for now just placeholder...
tab_b3["err_ly_b3"] = np.nan

B3_COLS = ["obs_id", "subrun", "ly_b3", "p_value", "err_ly_b3"]
tab_b3 = tab_b3[B3_COLS].drop_duplicates(subset=["obs_id", "subrun"])

# 1. RENAME columns in tab_b3 BEFORE merging
tab_b3 = tab_b3.rename(columns={"p_value": "ly_b3_p_value", "err_ly_b3": "ly_b3_err"})

print(f"[B3] {len(tab_b3):,} subrun rows loaded, "
      f"{tab_b3['obs_id'].nunique():,} unique runs.")

# 2. DROP any stale B3 columns based on tab_b3's current (renamed) columns
cols_to_drop = [c for c in tab_b3.columns if c not in ("obs_id", "subrun") and c in df_flat.columns]
df_flat = df_flat.drop(columns=cols_to_drop)

# 3. MERGE
df_flat = df_flat.merge(tab_b3, on=["obs_id", "subrun"], how="left")

n_matched = df_flat["ly_b3"].notna().sum()
print(f"[B3] Matched {n_matched:,} / {len(df_flat):,} subrun rows in df_flat.")

df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)
print(f"[B3] Saved enriched flat table -> {FNAME_DCHECK_FLAT_SRUNWISE}")

## Step 6 - Aggregate to a run-wise table and run-wise fit LightYields

Merges subrunwise `df_flat` into `df_flat_runwise` (runwise): durations/counts are summed, per-run constants (paths, cuts, PSF) take the first non-null value, everything else numeric gets a `corrected_elapsed_time` weighted mean, and the LYs rate gets a linear fit vs. time.

**In subrunwise table**
* ly_i_scaling, ly_b3_scaling + errors
  
**In runwise table (in addition to all that we have in subrunwise)**
* ly_i_fit_ (p0, u_p0, p1, u_p1, chi2, pvalue, ndf, error_flag)
* ly_b3_fit_ (p0, u_p0, p1, u_p1, chi2, pvalue, ndf, error_flag)

In [ ]:
%%time
df_flat = df_flat.rename(columns={
    "ly": "ly_i", "err_ly": "ly_i_err", "err_ly_b3": "ly_b3_err"})

# Duration/count columns -> summed to a run total
SUM_COLS = ["elapsed_time", "corrected_elapsed_time", "events",
            "ff_events", "pedestal_events", "num_contained_mu_rings"]
FIRST_COLS = ["dcheck_fname", "dl1_runwise_fname", "dl2_fname", "dl3_fname",
              "intensity_cut", "psf_spotsize", "psf_time", "time_since_psf_hours"]
ZD_COL = "zd"
EXCLUDE_FROM_WMEAN = set(SUM_COLS) | set(FIRST_COLS) | {
    ZD_COL, "obs_id", "subrun", "time", "dl1_srunwise_fname", "date_str", "ws_date_str",
}

# Everything else numeric — including ly_i, ly_i_err, ly_b3, ly_b3_err — gets a
# plain corrected_elapsed_time-weighted mean, same treatment as every other column.
WMEAN_COLS = [c for c in df_flat.columns
              if c not in EXCLUDE_FROM_WMEAN and pd.api.types.is_numeric_dtype(df_flat[c])]

def _weighted_mean_with_error(v, e, w):
    """Weighted mean of v (weights w), with error on the mean propagated in
    quadrature from e where available. If no per-point error is available
    (e.g. a not-yet-populated error column), the mean is still returned but
    its error comes back as NaN rather than being silently treated as 0."""
    v = np.asarray(v, dtype=float)
    e = np.asarray(e, dtype=float)
    w = np.asarray(w, dtype=float)

    mask_v = np.isfinite(v) & np.isfinite(w)
    if mask_v.sum() == 0:
        return np.nan, np.nan

    v_, w_ = v[mask_v], w[mask_v]
    mean = np.sum(w_ * v_) / w_.sum()

    mask_e = mask_v & np.isfinite(e)
    if mask_e.sum() == 0:
        return mean, np.nan   # value known, error simply not available yet

    e_, w_e = e[mask_e], w[mask_e]
    err = np.sqrt(np.sum((w_e * e_) ** 2)) / w_e.sum()
    return mean, err

# Minimum |p0 + p1*t| we trust as a denominator when turning the fit into a
# 1/fit scaling factor. If the fitted line comes anywhere near zero within
# the run's time span (which is exactly what happens when a single garbage
# point, e.g. ly_* ~ 1e9 in one subrun, drags the fit around), 1/fit_val
# diverges and both the scaling and its error explode. LY-like quantities
# are O(1), so a fitted value below this is already nonsensical.
MIN_FIT_VAL_FOR_SCALING = 1e-3

def _fit_and_scale(g, w, t, value_col, err_col, prefix, trim_edges=True,
                    min_value=1e-2, max_value=2.4):
    """Fit `value_col` vs time `t`, return fit-param dict (keys prefixed
    `{prefix}_fit_*`), the per-subrun 1/fit scaling Series (`{prefix}_scaling`),
    its error (`{prefix}_scaling_err`), and their weighted averages over the run
    (with the average error properly propagated, not just weighted-averaged).

    `trim_edges=True` drops the first and last (outlier-clipped) points of
    the run from the fit itself, for stability -- see `utils.fit_drdi`.

    Only points that `utils.fit_drdi` actually accepted (its `good_mask`,
    i.e. not clipped as outliers) get a scaling value at all: a point
    rejected as an outlier is rejected precisely because its value (and
    often its own error) can't be trusted, so letting it back in here would
    let one bad subrun (a ly_* of 1e9, or a NaN-adjacent error next to it)
    blow up `{prefix}_scaling_err` for the whole run.

    `err_col` may be entirely NaN (e.g. `ly_b3_err` before it's populated
    upstream) -- points are still included in the mask/fit as long as
    `value_col` itself is present; `utils.fit_drdi` falls back to an
    unweighted fit when errors aren't available, rather than dropping
    every point.
    """
    if value_col in g.columns:
        err_series = g[err_col] if err_col in g.columns else pd.Series(np.nan, index=g.index)
        mask = g[["corrected_elapsed_time", value_col]].notna().all(axis=1)
    else:
        err_series = pd.Series(np.nan, index=g.index)
        mask = pd.Series(False, index=g.index)

    mask_np = mask.to_numpy()
    good_np = np.zeros(len(g), dtype=bool)  # points fit_drdi actually trusted (subset of mask)

    if mask.sum() >= 2:
        # IMPORTANT: fit against the same `t` (real elapsed wall-clock time)
        # that gets used below (fit_val = p0 + p1 * t[...]) and in the sanity
        # plots (p0 + p1 * t_line). fit_drdi does NOT cumsum this internally
        # -- it must already be the x-axis, or the fit ends up calibrated to
        # a different x-scale than the one it's evaluated/plotted against,
        # which is what made the fitted line come out flat and off-trend.
        p0, p1, u_p0, u_p1, chi2_val, pval, ndf, err, good_sub = utils.fit_drdi(
            t[mask_np],
            g.loc[mask, value_col].to_numpy(dtype=float),
            err_series.loc[mask].to_numpy(dtype=float),
            trim_edges=trim_edges,
            min_value=min_value,
            max_value=max_value,
        )
        good_np[mask_np] = good_sub

    else:
        p0 = p1 = u_p0 = u_p1 = chi2_val = pval = ndf = np.nan
        err = True

    fit_params = {
        f"{prefix}_fit_p0": p0, f"{prefix}_fit_p1": p1,
        f"{prefix}_fit_u_p0": u_p0, f"{prefix}_fit_u_p1": u_p1,
        f"{prefix}_fit_chi2": chi2_val, f"{prefix}_fit_pvalue": pval,
        f"{prefix}_fit_ndf": ndf, f"{prefix}_fit_error_flag": err,
    }

    scaling = pd.Series(np.nan, index=g.index)
    scaling_err = pd.Series(np.nan, index=g.index)
    scaling_avg = np.nan
    scaling_err_avg = np.nan

    if mask.sum() >= 2 and np.isfinite(p0) and np.isfinite(p1) and good_np.any():
        fit_val = p0 + p1 * t[mask_np]
        with np.errstate(divide="ignore", invalid="ignore"):
            vals = 1.0 / fit_val
        # Guard against the fitted line coming close to (or crossing) zero:
        # a near-zero denominator, not just a non-finite one, is already an
        # unusable scaling factor and should be NaN, not a huge number.
        vals[np.abs(fit_val) < MIN_FIT_VAL_FOR_SCALING] = np.nan
        vals[~np.isfinite(vals)] = np.nan

        # Points fit_drdi clipped as outliers (good_np False) don't get a
        # scaling value at all, even at the per-subrun level.
        vals[~good_np[mask_np]] = np.nan
        scaling.loc[mask] = vals

        # Error propagation for S = 1/LY -> dS = dLY / LY^2
        # (vals**2 is equivalent to 1/LY^2). NaN err_col -> NaN scaling_err,
        # which is correct: we don't have an uncertainty to propagate yet.
        err_vals = err_series.loc[mask].to_numpy(dtype=float) * (vals ** 2)
        err_vals[~good_np[mask_np]] = np.nan
        scaling_err.loc[mask] = err_vals

        avg_mask = mask_np & good_np & scaling.notna().to_numpy()
        scaling_avg, scaling_err_avg = _weighted_mean_with_error(
            scaling[avg_mask], scaling_err[avg_mask], w[avg_mask]
        )

    return fit_params, scaling, scaling_avg, scaling_err, scaling_err_avg

def _aggregate_run(g: pd.DataFrame):
    g = g.sort_values("subrun")
    w = g["corrected_elapsed_time"].to_numpy(dtype=float)

    t_abs = pd.to_datetime(g["time"]).to_numpy()
    t = (t_abs - t_abs.min()) / np.timedelta64(1, "s")

    out = {"n_subruns": len(g), "time": g["time"].min()}
    out["zd"] = utils.weighted_mean(g[ZD_COL], w)
    out["min_zd"], out["max_zd"] = g[ZD_COL].min(), g[ZD_COL].max()
    for c in SUM_COLS:
        if c in g.columns:
            out[c] = g[c].sum(min_count=1)
    for c in FIRST_COLS:
        if c in g.columns:
            s = g[c].dropna()
            out[c] = s.iloc[0] if len(s) else np.nan
    for c in WMEAN_COLS:
        out[c] = utils.weighted_mean(g[c], w)

    # Fit, scale, and calculate scaling errors (independent of the ly_i/ly_b3
    # weighted means computed above via WMEAN_COLS)
    fit_ly_i, sc_ly_i, avg_sc_ly_i, err_ly_i, avg_err_ly_i = _fit_and_scale(g, w, t, "ly_i", "ly_i_err", "ly_i")
    fit_b3, sc_b3, avg_sc_b3, err_b3, avg_err_b3 = _fit_and_scale(
        g, w, t, "ly_b3", "ly_b3_err", "ly_b3",
        # ly_b3 has no error column yet (ly_b3_err is all-NaN), so the fit falls
        # back to unweighted OLS -- a single stray point drags the whole line.
        # The default sanity range (1e-2, 2.4) is tuned for ly_i and is too loose
        # for ly_b3 outliers (values like 0.02-0.09 slipped past min_value=1e-2).
        # Use the actual observed outlier range for ly_b3: real values cluster
        # near 1, so anything <0.1 or >10 is junk and gets excluded from the fit.
        min_value=1e-1, max_value=10,
    )

    out.update(fit_ly_i)
    out.update(fit_b3)
    out["ly_i_scaling"] = avg_sc_ly_i
    out["ly_i_scaling_err"] = avg_err_ly_i
    out["ly_b3_scaling"] = avg_sc_b3
    out["ly_b3_scaling_err"] = avg_err_b3

    return pd.Series(out), sc_ly_i, sc_b3, err_ly_i, err_b3

records = []
scaling_ly_i_parts, scaling_b3_parts = [], []
err_ly_i_parts, err_b3_parts = [], []

grouped = df_flat.groupby("obs_id", sort=True)
total_runs = len(grouped)
for i, (run, g) in enumerate(grouped, 1):
    print(f"Aggregating information for Run {run} ({i}/{total_runs})...", end="\r")
    row, sc_ly_i, sc_b3, err_ly_i, err_b3 = _aggregate_run(g)
    row["obs_id"] = run
    records.append(row)
    scaling_ly_i_parts.append(sc_ly_i)
    scaling_b3_parts.append(sc_b3)
    err_ly_i_parts.append(err_ly_i)
    err_b3_parts.append(err_b3)

# Push the per-subrun scaling factors and their errors back into df_flat
df_flat["ly_i_scaling"] = pd.concat(scaling_ly_i_parts).reindex(df_flat.index)
df_flat["ly_i_scaling_err"] = pd.concat(err_ly_i_parts).reindex(df_flat.index)
df_flat["ly_b3_scaling"] = pd.concat(scaling_b3_parts).reindex(df_flat.index)
df_flat["ly_b3_scaling_err"] = pd.concat(err_b3_parts).reindex(df_flat.index)

df_flat_runwise = pd.DataFrame(records)
front_cols = ["obs_id", "n_subruns", "time", "elapsed_time", "corrected_elapsed_time",
              "zd", "min_zd", "max_zd", "az", "ra", "dec", "events", "ly_i", "ly_i_err"]

fit_cols = [c for c in df_flat_runwise.columns
            if c.startswith("ly_i_fit_") or c.startswith("ly_b3_fit_")
            or c in ("ly_i_scaling", "ly_b3_scaling", "ly_i_scaling_err", "ly_b3_scaling_err")]

other_cols = [c for c in df_flat_runwise.columns if c not in front_cols + fit_cols]
df_flat_runwise = df_flat_runwise[front_cols + fit_cols + other_cols]
df_flat_runwise = df_flat_runwise.sort_values("obs_id", ignore_index=True)

df_flat_runwise.to_parquet(FNAME_DCHECK_FLAT_RUNWISE, index=False)
df_flat.to_parquet(FNAME_DCHECK_FLAT_SRUNWISE, index=False)

print(f"Saved run-wise table: {len(df_flat_runwise):,} runs -> {FNAME_DCHECK_FLAT_RUNWISE}")
df_flat_runwise.head(3)

In [ ]:
N_SANITY_RUNS = 24
RNG_SEED = 40
rng = np.random.default_rng(RNG_SEED)

# Only consider runs where at least one of the two fits actually succeeded
# (error_flag False and params not NaN) -- skip runs that were all-NaN.
has_valid_fit = (
    (df_flat_runwise["ly_i_fit_error_flag"] == False) & df_flat_runwise["ly_i_fit_p0"].notna()
) | (
    (df_flat_runwise["ly_b3_fit_error_flag"] == False) & df_flat_runwise["ly_b3_fit_p0"].notna()
)
valid_runs = df_flat_runwise.loc[has_valid_fit, "obs_id"].to_numpy()

if len(valid_runs) == 0:
    print("No runs with a valid fit were found -- nothing to sanity check.")
else:
    n_pick = min(N_SANITY_RUNS, len(valid_runs))
    sample_runs = rng.choice(valid_runs, size=n_pick, replace=False)

    ncols = 4
    nrows = int(np.ceil(n_pick / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    series_specs = [
        ("ly_i", "ly_i_err", "ly_i", "tab:blue"),
        ("ly_b3", "ly_b3_err", "ly_b3", "tab:orange"),
    ]

    for ax, run in zip(axes_flat, sample_runs):
        g = df_flat.loc[df_flat["obs_id"] == run].sort_values("subrun")
        t_abs = pd.to_datetime(g["time"]).to_numpy()
        t = (t_abs - t_abs.min()) / np.timedelta64(1, "s")

        row = df_flat_runwise.loc[df_flat_runwise["obs_id"] == run].iloc[0]

        any_plotted = False
        annotation_lines = []
        for value_col, err_col, prefix, color in series_specs:
            if value_col not in g.columns or err_col not in g.columns:
                continue
            mask = g[[value_col, err_col]].notna().all(axis=1).to_numpy()
            if mask.sum() == 0:
                continue

            ax.errorbar(t[mask], g.loc[mask, value_col].to_numpy(),
                        yerr=g.loc[mask, err_col].to_numpy(),
                        fmt="o", ms=3, color=color, alpha=0.6, label=f"{value_col} data")
            any_plotted = True

            p0, p1 = row.get(f"{prefix}_fit_p0"), row.get(f"{prefix}_fit_p1")
            err_flag = row.get(f"{prefix}_fit_error_flag")
            chi2, ndf, pval = (row.get(f"{prefix}_fit_chi2"),
                                row.get(f"{prefix}_fit_ndf"),
                                row.get(f"{prefix}_fit_pvalue"))

            if pd.notna(p0) and pd.notna(p1) and not err_flag:
                t_line = np.linspace(t[mask].min(), t[mask].max(), 50)
                ax.plot(t_line, p0 + p1 * t_line, "-", color=color, label=f"{value_col} fit")
                annotation_lines.append(
                    f"{value_col}: chi2/ndf={chi2:.2f}/{ndf:.0f}  p={pval:.2f}"
                    if pd.notna(chi2) and pd.notna(ndf) and pd.notna(pval)
                    else f"{value_col}: fit ok (no chi2 info)"
                )
            else:
                annotation_lines.append(f"{value_col}: fit failed/NaN")

        if not any_plotted:
            ax.text(0.5, 0.5, "no valid data", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.text(0.02, 0.02, "\n".join(annotation_lines), transform=ax.transAxes,
                     fontsize=6, va="bottom", ha="left",
                     bbox=dict(boxstyle="round", fc="white", alpha=0.7, ec="gray"))

        ax.set_title(f"Run {run}", fontsize=10)
        ax.set_xlabel("t [s]")
        ax.legend(fontsize=6, loc="upper right")

    for ax in axes_flat[n_pick:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
N_SANITY_RUNS = 24
RNG_SEED = 40
rng = np.random.default_rng(RNG_SEED)

# Only consider runs where the ly_b3 fit actually succeeded
# (error_flag False and params not NaN) -- skip runs that were all-NaN for ly_b3.
has_valid_b3_fit = (
    (df_flat_runwise["ly_b3_fit_error_flag"] == False) & df_flat_runwise["ly_b3_fit_p0"].notna()
)
valid_runs = df_flat_runwise.loc[has_valid_b3_fit, "obs_id"].to_numpy()

if len(valid_runs) == 0:
    print("No runs with a valid ly_b3 fit were found -- nothing to sanity check.")
else:
    n_pick = min(N_SANITY_RUNS, len(valid_runs))
    sample_runs = rng.choice(valid_runs, size=n_pick, replace=False)

    ncols = 4
    nrows = int(np.ceil(n_pick / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 3.5 * nrows), squeeze=False)
    axes_flat = axes.ravel()

    for ax, run in zip(axes_flat, sample_runs):
        g = df_flat.loc[df_flat["obs_id"] == run].sort_values("subrun")
        t_abs = pd.to_datetime(g["time"]).to_numpy()
        t = (t_abs - t_abs.min()) / np.timedelta64(1, "s")

        row = df_flat_runwise.loc[df_flat_runwise["obs_id"] == run].iloc[0]

        any_plotted = False
        annotation_lines = []

        # --- background: ly_i, plotted in light gray for context (no annotation) ---
        if "ly_i" in g.columns and "ly_i_err" in g.columns:
            mask_i = g[["ly_i", "ly_i_err"]].notna().all(axis=1).to_numpy()
            if mask_i.sum() > 0:
                ax.errorbar(t[mask_i], g.loc[mask_i, "ly_i"].to_numpy(),
                            yerr=g.loc[mask_i, "ly_i_err"].to_numpy(),
                            fmt="o", ms=3, color="0.7", alpha=0.5,
                            label="ly_i data", zorder=1)

                p0_i, p1_i = row.get("ly_i_fit_p0"), row.get("ly_i_fit_p1")
                err_flag_i = row.get("ly_i_fit_error_flag")
                if pd.notna(p0_i) and pd.notna(p1_i) and not err_flag_i:
                    t_line_i = np.linspace(t[mask_i].min(), t[mask_i].max(), 50)
                    ax.plot(t_line_i, p0_i + p1_i * t_line_i, "--",
                            color="k", alpha=0.8, linewidth=1,
                            label="ly_i fit", zorder=100)

        # --- foreground: ly_b3, the series we're actually sanity-checking ---
        value_col, err_col, prefix, color = "ly_b3", "ly_b3_err", "ly_b3", "tab:orange"
        if value_col in g.columns and err_col in g.columns:
            mask = g[[value_col, err_col]].notna().all(axis=1).to_numpy()
            if mask.sum() > 0:
                ax.errorbar(t[mask], g.loc[mask, value_col].to_numpy(),
                            yerr=g.loc[mask, err_col].to_numpy(),
                            fmt="o", ms=3, color=color, alpha=0.8,
                            label=f"{value_col} data", zorder=2)
                any_plotted = True

                p0, p1 = row.get(f"{prefix}_fit_p0"), row.get(f"{prefix}_fit_p1")
                err_flag = row.get(f"{prefix}_fit_error_flag")
                chi2, ndf, pval = (row.get(f"{prefix}_fit_chi2"),
                                    row.get(f"{prefix}_fit_ndf"),
                                    row.get(f"{prefix}_fit_pvalue"))

                if pd.notna(p0) and pd.notna(p1) and not err_flag:
                    t_line = np.linspace(t[mask].min(), t[mask].max(), 50)
                    ax.plot(t_line, p0 + p1 * t_line, "-", color=color,
                            linewidth=2, label=f"{value_col} fit", zorder=2)
                    annotation_lines.append(
                        f"{value_col}: chi2/ndf={chi2:.2f}/{ndf:.0f}  p={pval:.2f}"
                        if pd.notna(chi2) and pd.notna(ndf) and pd.notna(pval)
                        else f"{value_col}: fit ok (no chi2 info)"
                    )
                else:
                    annotation_lines.append(f"{value_col}: fit failed/NaN")

        if not any_plotted:
            ax.text(0.5, 0.5, "no valid ly_b3 data", ha="center", va="center", transform=ax.transAxes)
        else:
            ax.text(0.02, 0.02, "\n".join(annotation_lines), transform=ax.transAxes,
                     fontsize=6, va="bottom", ha="left",
                     bbox=dict(boxstyle="round", fc="white", alpha=0.7, ec="gray"))

        ax.set_title(f"Run {run}", fontsize=10)
        ax.set_xlabel("t [s]")
        ax.legend(fontsize=6, loc="upper right")

    for ax in axes_flat[n_pick:]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
df_flat_runwise.columns.tolist()